In [ ]:
# Silver 변환 노트북: Fear & Greed Bronze → Silver
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark.sql("SET spark.sql.session.timeZone=UTC")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"
BRONZE  = f"{CATALOG}.{SCHEMA}.bronze_fear_greed"
SILVER  = f"{CATALOG}.{SCHEMA}.silver_fear_greed"

# ===== Spark / Delta Performance Configuration =====
# optimizeWrite: 커밋 전 소파일을 병합 → 소파일 누적 방지
spark.conf.set("spark.databricks.delta.optimizeWrite", "true")
# autoCompact: 쓰기 완료 후 백그라운드 컴팩션 자동 트리거
spark.conf.set("spark.databricks.delta.autoCompact", "true")
# AQE: 런타임 실행 통계를 기반으로 Spark가 쿼리 플랜을 동적으로 재조정
# FNG는 sparse 데이터(하루 1건)이므로 AQE가 빈 셔플 파티션을 효과적으로 병합
spark.conf.set("spark.sql.adaptive.enabled", "true")
# 셔플 후 소규모/빈 파티션을 동적으로 병합하여 Task 오버헤드 감소
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Sort-Merge Join에서 데이터 Skew를 자동 감지하고 서브태스크로 분할
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Silver 스키마(호환용으로 ts_unix 포함)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER} (
  ts_unix              BIGINT,      -- event_time(초) 파생
  event_time           TIMESTAMP,   -- 원본 UTC 시각
  dt                   DATE,        -- 파티션
  index_value          INT,         -- 0~100
  value_classification STRING,      -- "Extreme Fear" ~ "Extreme Greed"
  time_until_update    STRING,      -- API 텍스트
  unique_key           STRING       -- 'fear_greed|<unix_ts>'
) USING DELTA
PARTITIONED BY (dt)
""")

src = spark.table(BRONZE)

silver_df = (
  src.select(
      F.col("event_time").cast("timestamp").alias("event_time"),
      F.col("dt").cast("date").alias("dt"),
      F.col("index_value").cast("int").alias("index_value"),
      F.col("value_classification").alias("value_classification"),
      F.col("time_until_update").alias("time_until_update"),
      F.col("unique_key").alias("unique_key")
  )
  .withColumn("ts_unix", F.unix_timestamp("event_time").cast("long"))
  .dropDuplicates(["unique_key"])
  .repartition("dt")
)

t = DeltaTable.forName(spark, SILVER)
# updateAll을 써도 s에 ts_unix가 있으므로 안전
(t.alias("t")
 .merge(silver_df.alias("s"), "t.unique_key = s.unique_key AND t.dt = s.dt")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

print(f"[FNG SILVER] upsert complete -> {SILVER}")

In [ ]:
# ===== DATA QUALITY CHECKS (silver_fear_greed) =====
# MERGE 완료 후 전체 테이블에 대해 FNG 데이터 계약 검증 수행
# Fear & Greed Index는 항상 0~100 범위여야 하며 일별 고유성 보장
from pyspark.sql import functions as F

dq_fng = spark.table(SILVER)

fng_check = dq_fng.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("index_value").isNull().cast("int")).alias("null_value"),
    F.sum((F.col("index_value") < 0).cast("int")).alias("below_range"),
    F.sum((F.col("index_value") > 100).cast("int")).alias("above_range"),
    F.sum(F.col("value_classification").isNull().cast("int")).alias("null_classification"),
    F.countDistinct("dt").alias("distinct_days"),
).collect()[0]

print(f"[DQ] silver_fear_greed")
print(f"  total_rows          : {fng_check['total_rows']}")
print(f"  null_value          : {fng_check['null_value']}")
print(f"  범위 이탈 (0~100)   : 미만={fng_check['below_range']}건, 초과={fng_check['above_range']}건")
print(f"  null_classification : {fng_check['null_classification']}")
print(f"  distinct_days       : {fng_check['distinct_days']}")

# 핵심 품질 계약 위반 시 파이프라인 즉시 중단
assert fng_check["null_value"] == 0, \
    f"[DQ FAIL] index_value에 null 발생: {fng_check['null_value']}건"
assert fng_check["below_range"] == 0 and fng_check["above_range"] == 0, \
    f"[DQ FAIL] Fear & Greed Index가 0-100 범위 이탈: 미만={fng_check['below_range']}, 초과={fng_check['above_range']}"
assert fng_check["null_classification"] == 0, \
    f"[DQ FAIL] value_classification에 null 발생: {fng_check['null_classification']}건"

print("[DQ] 모든 품질 체크 통과 ✓")